# Q-Refine: Interactive Demo

Welcome to the Q-Refine interactive developer notebook. 
Run the cells below to generate a Quantum Neural Network, apply hardware noise, and mitigate it using Zero-Noise Extrapolation (ZNE).

In [ ]:
from q_refine.circuits.qnn import generate_trained_qnn
from q_refine.mitigation_engine.q_sanitizer import QSanitizer
from q_refine.benchmark_engine.hardware_profiler import HardwareProfiler
from q_refine.core.dashboard import QRefineDashboard
from q_refine_pipeline import get_success_probability

# 1. Profile Hardware
profiler = HardwareProfiler(use_real_hardware=False)
noise_model = profiler.get_noise_model()

# 2. Generate Circuit
circuit = generate_trained_qnn(num_qubits=5, num_layers=2)
secret = "00000"
raw_prob = get_success_probability(circuit, secret, noise_model)
print(f"Raw Unmitigated Probability: {raw_prob:.2%}")

In [ ]:
# 3. Mitigate with ZNE
sanitizer = QSanitizer(mitigation_method="ZNE")
folded_circuits = sanitizer.refine(circuit)

scales, noisy_probs = [], []
for scale, folded_circ in folded_circuits.items():
    prob = get_success_probability(folded_circ, secret, noise_model)
    scales.append(scale)
    noisy_probs.append(prob)

mitigated_prob = sanitizer.richardson_extrapolate(noisy_probs, scales)
print(f"Mitigated Probability: {mitigated_prob:.2%}")

In [ ]:
# 4. View Dashboard Inline
QRefineDashboard.generate_report(raw_prob, mitigated_prob, scales, noisy_probs, "demo_dashboard.png")
from IPython.display import Image
Image(filename="demo_dashboard.png")